# Dónde construir: estaciones para una alcaldía nueva

## Contexto

Una nueva administración acaba de asumir la Alcaldía de Medellín. Dentro del plan de desarrollo se aprobó una partida para construir estaciones que atiendan tres tipos de incidentes:

| Tipo de incidente | Estación que lo atiende |
|---|---|
| Hurto | CAI de policía |
| Accidente de tránsito | Estación de tránsito y grúas |
| Incendio o emergencia | Estación de bomberos |

La Secretaría de Planeación entrega un histórico de incidentes de los últimos años, la red vial de la ciudad y una lista de lotes disponibles donde se podría construir. Cada estación tiene un costo que depende de su tipo y de la zona del lote, y solo atiende con oportunidad los incidentes de su tipo que estén a cierta distancia por las calles.

El presupuesto no alcanza para cubrir toda la ciudad. Tu equipo debe decidir **qué estaciones construir y en qué lotes**, de manera que la ciudad quede lo mejor atendida posible sin exceder el presupuesto, y presentar la propuesta al concejo con un mapa, el costo y los casos que se cubrirían.

La red vial corresponde a las calles reales de Medellín. El histórico de incidentes, los lotes y los costos son simulados para el curso.

## Los datos

Cada instancia está en una carpeta con cinco archivos:

- **`nodos.csv`**: intersecciones y puntos de la red. Columnas `id`, `lat`, `lon`.
- **`aristas.csv`**: tramos de calle. Columnas `origen`, `destino`, `longitud_m`, `nombre`. Cada fila es un tramo que se puede recorrer **solo de `origen` a `destino`**. Una calle de doble sentido aparece dos veces, una por cada dirección. Algunos tramos no tienen nombre.
- **`lotes.csv`**: lotes disponibles. Columnas `lote` (código), `nodo` (intersección donde está el lote) y `factor_zona`.
- **`incidentes.csv`**: histórico. Columnas `id`, `nodo` (intersección donde ocurrió), `tipo` (`hurto`, `accidente` o `incendio`) y `fecha_hora`.
- **`parametros.json`**: presupuesto total y, para cada tipo de incidente, el nombre de la estación, su costo base, su radio de cobertura y su peso de prioridad.

Los parámetros de las tres instancias son los mismos salvo el presupuesto:

| Tipo | Estación | Costo base (millones) | Radio de cobertura | Peso |
|---|---|---|---|---|
| hurto | CAI | 1.000 | 1.000 m | 1 |
| accidente | Tránsito y grúas | 2.000 | 2.000 m | 3 |
| incendio | Bomberos | 5.000 | 3.000 m | 5 |

### Las tres instancias

| Instancia | Descripción | Nodos | Tramos | Incidentes | Lotes | Presupuesto (millones) |
|---|---|---|---|---|---|---|
| `pequena` | Barrio ficticio | 9 | 22 | 18 | 5 | 6.000 |
| `comuna` | Recorte de la red real alrededor de Laureles | 2.893 | 6.782 | 913 | 15 | 12.000 |
| `ciudad` | Red vial completa de Medellín | 27.568 | 67.330 | 5.300 | 80 | 40.000 |

### Descripción del barrio de la instancia `pequena`

El barrio es una cuadrícula de 3 × 3 intersecciones numeradas de 0 a 8. Vista como un mapa con el norte hacia arriba:

```
 6 ── 7 ── 8        fila de arriba:  Calle 3
 │    │    ↑
 3 ── 4 →─ 5        fila del medio:  Calle 2
 │    │    │
 0 ── 1 ── 2        fila de abajo:   Calle 1

Carrera A (0-3-6)   Carrera B (1-4-7)   Carrera C (2-5-8)
```

Todas las calles son de doble sentido excepto dos tramos: en la Calle 2 solo se puede ir **de 4 a 5**, y en la Carrera C solo se puede ir **de 5 a 8**. Cada tramo mide entre 900 m y 990 m; las longitudes exactas están en `aristas.csv`.

Los lotes disponibles están en las intersecciones 1, 3, 5, 6 y 8:

| lote | nodo | factor_zona |
|---|---|---|
| L1 | 1 | 1.1 |
| L2 | 3 | 1.5 |
| L3 | 5 | 1.4 |
| L4 | 6 | 1.5 |
| L5 | 8 | 1.5 |

Los 18 incidentes del histórico se distribuyen así (H = hurtos, A = accidentes, I = incendios):

| Nodo | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| Incidentes | — | H1 | H1 A2 I1 | H3 | H2 A1 | A1 | A1 I1 | H1 | H1 A1 I1 |

## Reglas del modelo

1. **Costo.** Construir una estación de un tipo en un lote cuesta `costo_base del tipo × factor_zona del lote`, redondeado al millón más cercano. Por ejemplo, un CAI en el lote L1 de la instancia `pequena` cuesta 1.000 × 1.1 = 1.100 millones.
2. **Un lote, una estación.** En cada lote se construye a lo sumo una estación. Sí se pueden construir varias estaciones del mismo tipo en lotes distintos.
3. **Presupuesto.** La suma de los costos de las estaciones construidas no puede superar el presupuesto de la instancia.
4. **Cobertura.** Un incidente queda cubierto si existe una estación **de su mismo tipo** tal que la distancia más corta por la red vial **desde el nodo de la estación hasta el nodo del incidente** es menor o igual al radio de cobertura de ese tipo. La distancia se mide respetando el sentido de los tramos, porque es el vehículo de la estación el que se desplaza hasta el incidente.
5. **Valor de una propuesta.** Es la suma de los pesos de los incidentes cubiertos. Un incidente cubierto por varias estaciones cuenta una sola vez. Por ejemplo, una propuesta que cubre 10 hurtos, 4 accidentes y 1 incendio vale 10 × 1 + 4 × 3 + 1 × 5 = 27.

### Supuestos

- La red vial es una aproximación: se retiraron ciclorrutas, senderos y pasos peatonales identificables por su nombre.
- Los vehículos de emergencia pueden circular por los carriles de Metroplús, que hacen parte de la red.
- Todos los incidentes de un mismo tipo tienen la misma prioridad, sin importar su fecha.

## Restricciones de implementación

- La solución se escribe en Python.
- Se pueden usar `csv`, `json` o `pandas` **solo para leer los archivos**, y `heapq`.
- Para el mapa y la interfaz gráfica se puede usar cualquier librería de visualización o de interfaces (por ejemplo `folium`, `matplotlib`, `streamlit` o `tkinter`), siempre que no se use para calcular rutas, distancias ni combinaciones.
- No se permite usar `networkx`, `osmnx`, `scipy`, `itertools` ni ninguna librería que calcule rutas, distancias en grafos o combinaciones.

## Cronograma y entregables

| Semana | Entrega |
|---|---|
| 1 | Parte 1 |
| 2 | Parte 2 |
| 3 | Parte 3 |
| 4 | Parte 4 y sustentación |

Cada entrega se hace sobre este mismo notebook. Las partes posteriores pueden reutilizar el código de las anteriores.

### Sustentación

En la sustentación se entregará a cada equipo una nueva distribución de lotes y un nuevo histórico de incidentes, con el mismo formato de archivos descrito en la sección "Los datos". El equipo deberá ejecutar su solución sobre esos datos durante la sustentación y ajustarla si es necesario, así que conviene que el código lea cualquier instancia desde sus archivos y no dependa de valores particulares de las instancias entregadas.

### Nota: feria de proyectos de ingeniería

Los equipos que quieran presentar esta práctica en la feria de proyectos de ingeniería deben, además de todo lo anterior, desarrollar una **aplicación interactiva con interfaz gráfica de usuario**, por ejemplo una que permita elegir la instancia, cambiar el presupuesto o los pesos de prioridad y ver la propuesta resultante sobre el mapa. Sin esta aplicación el proyecto no podrá competir.

El equipo ganador de la feria quedará **eximido del examen final**.

## Parte 1. Alcance de cada estación posible

Para una instancia cualquiera, calcula para **cada lote y cada tipo de estación**:

- el costo de construir esa estación en ese lote, y
- la lista de `id` de los incidentes que esa estación cubriría por sí sola, según la regla 4.

### Entrada

El nombre de una instancia (`pequena`, `comuna` o `ciudad`). El programa lee los cinco archivos de su carpeta.

### Salida

Para cada par (lote, tipo): su costo en millones y la lista ordenada de `id` de los incidentes que cubre.

### Restricción adicional

El cálculo para la instancia `ciudad`, con sus 80 lotes y 3 tipos, debe terminar en menos de un minuto.

In [ ]:
casos_ej_001_parte1 = [
    {
        "entrada": {"instancia": "pequena", "lote": "L3", "tipo": "accidente"},
        "salida": {"costo": 2800, "incidentes_cubiertos": [1, 2, 4, 6]},
    },
    {
        "entrada": {"instancia": "pequena", "lote": "L2", "tipo": "hurto"},
        "salida": {"costo": 1500, "incidentes_cubiertos": [9, 10, 11, 12, 13]},
    },
]

### Solución de la Parte 1

Todo el código de la solución vive en el paquete `estaciones/` de esta misma carpeta; estas celdas solo lo llaman. Así el notebook, la línea de comandos (`py main.py ...`) y la interfaz de la feria usan exactamente el mismo código y no hay copias que se desincronicen.

Cómo se calcula cada par (lote, tipo):

- **Costo**: `costo_base × factor_zona`, redondeado al millón más cercano con `.5` hacia arriba (`decimal`, exacto).
- **Cobertura**: un Dijkstra (con `heapq`) por lote, **respetando el sentido de las calles**, que se detiene al pasar el radio más grande (3.000 m). Las distancias se guardan como enteros (por ejemplo, en décimas de metro) para que la comparación `distancia <= radio` sea exacta. Esas distancias sirven para los tres tipos de estación del lote.

La primera celda prepara el entorno: busca la carpeta del proyecto (la que contiene `estaciones/`) a partir del directorio actual, la agrega a `sys.path` e importa el paquete.

In [1]:
import sys
import time
from pathlib import Path


def buscar_raiz_del_proyecto():
    """Carpeta que contiene el paquete `estaciones` (la del notebook o una superior).

    Jupyter y VS Code ejecutan el notebook con el directorio actual en la
    carpeta del notebook, que es la raíz del proyecto.
    """
    carpeta = Path.cwd().resolve()
    candidatas = [carpeta]
    for superior in carpeta.parents:
        candidatas.append(superior)
    for candidata in candidatas:
        if (candidata / "estaciones" / "__init__.py").exists():
            return candidata
    raise FileNotFoundError(
        "No se encontró la carpeta `estaciones/`. Abre este notebook desde la carpeta "
        "del proyecto (la que contiene main.py, estaciones/ e instancias/).")


RAIZ = buscar_raiz_del_proyecto()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from estaciones.exacto import resolver
from estaciones.heuristicas import base, propia, propia_ls
from estaciones.lectura import cargar_instancia, carpeta_de_instancia
from estaciones.mapa import exportar_html, leer_fechas, tabla_texto
from estaciones.modelo import Modelo

print("Proyecto encontrado: la carpeta", RAIZ.name, "contiene estaciones/")

Proyecto encontrado: la carpeta practica contiene estaciones/


In [2]:
MODELOS = {}
SEGUNDOS_DE_PRECALCULO = {}


def modelo_de(instancia):
    """Modelo de la instancia (lectura + costos + coberturas), calculado una sola vez.

    Se guarda cuánto tardó: es el precálculo de la Parte 1 y no se suma
    al tiempo de las heurísticas ni del óptimo.
    """
    if instancia not in MODELOS:
        inicio = time.perf_counter()
        MODELOS[instancia] = Modelo(cargar_instancia(instancia))
        SEGUNDOS_DE_PRECALCULO[instancia] = time.perf_counter() - inicio
    return MODELOS[instancia]


def parte1(entrada):
    """Costo y lista ordenada de incidentes que cubre la estación (lote, tipo)."""
    modelo = modelo_de(entrada["instancia"])
    return {
        "costo": modelo.costo(entrada["lote"], entrada["tipo"]),
        "incidentes_cubiertos": modelo.cobertura(entrada["lote"], entrada["tipo"]),
    }


for caso in casos_ej_001_parte1:
    obtenido = parte1(caso["entrada"])
    assert obtenido == caso["salida"], (caso["entrada"], obtenido)
print("OK: los", len(casos_ej_001_parte1), "casos de ejemplo de la Parte 1 coinciden")

OK: los 2 casos de ejemplo de la Parte 1 coinciden


In [3]:
modelo = modelo_de("pequena")
print(f"{'lote':<6}{'tipo':<11}{'costo':>7}  incidentes cubiertos")
for codigo_lote, tipo in modelo.todas_las_opciones():
    costo = modelo.costo(codigo_lote, tipo)
    cubiertos = modelo.cobertura(codigo_lote, tipo)
    print(f"{codigo_lote:<6}{tipo:<11}{costo:>7}  {cubiertos}")

lote  tipo         costo  incidentes cubiertos
L1    hurto         1100  [7, 8, 12, 13]
L1    accidente     2200  [1, 2, 3, 4]
L1    incendio      5500  [16, 17, 18]
L2    hurto         1500  [9, 10, 11, 12, 13]
L2    accidente     3000  [3, 4, 5]
L2    incendio      7500  [16, 17, 18]
L3    hurto         1400  [8, 15]
L3    accidente     2800  [1, 2, 4, 6]
L3    incendio      7000  [16, 17, 18]
L4    hurto         1500  [9, 10, 11, 14]
L4    accidente     3000  [3, 5, 6]
L4    incendio      7500  [17, 18]
L5    hurto         1500  [14, 15]
L5    accidente     3000  [3, 5, 6]
L5    incendio      7500  [17, 18]


In [4]:
for instancia in ("comuna", "ciudad"):
    inicio = time.perf_counter()
    modelo = modelo_de(instancia)
    pares = 0
    for codigo_lote, tipo in modelo.todas_las_opciones():
        modelo.costo(codigo_lote, tipo)
        modelo.cobertura(codigo_lote, tipo)
        pares += 1
    segundos = time.perf_counter() - inicio
    print(f"{instancia}: {len(modelo.instancia.lotes)} lotes, {pares} pares (lote, tipo), "
          f"{segundos:.2f} s en total (lectura + Dijkstra + costos)")
print("Requisito: ciudad en menos de un minuto.")

comuna: 15 lotes, 45 pares (lote, tipo), 0.07 s en total (lectura + Dijkstra + costos)
ciudad: 80 lotes, 240 pares (lote, tipo), 0.66 s en total (lectura + Dijkstra + costos)
Requisito: ciudad en menos de un minuto.


## Parte 2. Propuestas construidas paso a paso

### 2.1 Heurística base

La Secretaría de Planeación usa hoy el siguiente procedimiento para armar una propuesta:

1. Empezar sin estaciones y con todo el presupuesto disponible.
2. Considerar todas las estaciones posibles (lote, tipo) que cumplen dos condiciones: el lote todavía está libre y su costo no supera el presupuesto restante.
3. De ellas, elegir la que **agrega más valor**, es decir, la que tiene mayor suma de pesos de los incidentes que cubre y que **todavía no están cubiertos** por las estaciones ya elegidas. Si hay empate, se elige la de menor código de lote; si el empate es dentro del mismo lote, se prefiere en el orden hurto, accidente, incendio.
4. Si ninguna estación posible agrega valor, terminar. Si no, construirla, descontar su costo, marcar el lote como ocupado y volver al paso 2.

Implementa la heurística base y aplícala a las tres instancias.

### 2.2 Heurísticas propias

Diseña al menos una heurística propia que también construya la propuesta eligiendo una estación a la vez, y que obtenga un valor mayor que la base en al menos una de las tres instancias. Aplícala a las tres instancias.

Para cada heurística, la base incluida, construye a mano una instancia pequeña (en el mismo formato de archivos, con no más de 6 nodos) en la que la heurística **no** obtenga la mejor propuesta posible. Explica qué decisión de la heurística la lleva por mal camino en ese caso.

### Salida

Para cada heurística e instancia: las estaciones elegidas (lote y tipo, ordenadas por código de lote), el costo total, la cantidad de casos cubiertos de cada tipo y el valor de la propuesta.

In [ ]:
casos_ej_001_parte2 = [
    {
        "entrada": {"instancia": "pequena", "heuristica": "base"},
        "salida": {
            "estaciones": [("L1", "incendio")],
            "costo_total": 5500,
            "casos_cubiertos": {"hurto": 0, "accidente": 0, "incendio": 3},
            "valor": 15,
        },
    },
    {
        "entrada": {"instancia": "comuna", "heuristica": "base"},
        "salida": {
            "estaciones": [
                ("L05", "accidente"), ("L06", "hurto"), ("L08", "accidente"),
                ("L09", "incendio"), ("L13", "hurto"), ("L14", "hurto"),
                ("L15", "accidente"),
            ],
            "costo_total": 11950,
            "casos_cubiertos": {"hurto": 486, "accidente": 296, "incendio": 31},
            "valor": 1529,
        },
    },
]

### Solución de la Parte 2

Las tres heurísticas usan la misma función `goloso` (en `estaciones/heuristicas.py`): en cada paso miran las estaciones con el lote libre y que caben en el presupuesto restante, calculan el **valor nuevo** de cada una (pesos de los incidentes que cubre y que aún no están cubiertos) y eligen la de mayor *puntaje*. Solo cambia el puntaje:

- **`base`** (2.1): puntaje = valor nuevo. Las opciones se recorren por código de lote en orden natural y, dentro del lote, hurto, accidente, incendio; solo un puntaje *estrictamente* mayor reemplaza a la mejor, así el empate lo gana la primera, que es el desempate del enunciado.
- **`propia`** (2.2): puntaje = valor nuevo **por millón** (`valor_nuevo / costo`). Construye una estación a la vez, igual que la base, pero premia las estaciones que rinden más por cada peso. Es la heurística con la que se cumple el requisito 2.2: supera a la base en `pequena` (19 > 15) y en `ciudad` (7706 > 7591); en `comuna` pierde (1492 < 1529).
- **`propia_ls`**: es **híbrida**, no puramente constructiva. Primero hace varios arranques golosos de una estación a la vez (los dos golosos anteriores y uno por cada posible primera estación, completando con la razón) y se queda con el mejor; después aplica **mejora local**: cambia una estación por otra, quita una o dos, o quita dos y pone una elegida, y rellena el hueco con el goloso. Salta al mejor vecino mientras alguno mejore. Es la de mayor valor en las tres instancias, pero en `ciudad` tarda entre 20 y 35 s según el equipo (las otras dos, centésimas de segundo).

Las funciones `parte2(entrada)` y `parte3(entrada)` devuelven exactamente las cuatro claves del enunciado (`estaciones` como lista de tuplas ordenada por código de lote, `costo_total`, `casos_cubiertos` con hurto, accidente e incendio, y `valor`), sin claves extra.

In [5]:
HEURISTICAS = {"base": base, "propia": propia, "propia_ls": propia_ls}


def salida_del_enunciado(resultado):
    """Solo las cuatro claves que pide el enunciado, con los tipos de sus ejemplos."""
    estaciones = []
    for codigo_lote, tipo in resultado["estaciones"]:
        estaciones.append((codigo_lote, tipo))
    casos = {}
    for tipo in ("hurto", "accidente", "incendio"):
        casos[tipo] = resultado["casos_cubiertos"].get(tipo, 0)
    return {
        "estaciones": estaciones,
        "costo_total": resultado["costo_total"],
        "casos_cubiertos": casos,
        "valor": resultado["valor"],
    }


def parte2(entrada):
    """Propuesta de la heurística pedida ("base", "propia" o "propia_ls")."""
    heuristica = HEURISTICAS[entrada["heuristica"]]
    return salida_del_enunciado(heuristica(modelo_de(entrada["instancia"])))


for caso in casos_ej_001_parte2:
    obtenido = parte2(caso["entrada"])
    assert obtenido == caso["salida"], (caso["entrada"], obtenido)
print("OK: los", len(casos_ej_001_parte2), "casos de ejemplo de la Parte 2 coinciden")

OK: los 2 casos de ejemplo de la Parte 2 coinciden


In [6]:
RESULTADOS = {}
for instancia in ("pequena", "comuna", "ciudad"):
    modelo = modelo_de(instancia)
    print(f"===== {instancia} (presupuesto {modelo.presupuesto})")
    for nombre, heuristica in HEURISTICAS.items():
        inicio = time.perf_counter()
        resultado = heuristica(modelo)
        segundos = time.perf_counter() - inicio
        RESULTADOS[(instancia, nombre)] = (resultado, segundos)
        salida = salida_del_enunciado(resultado)
        print(f"{nombre}:")
        print(f"  estaciones={salida['estaciones']}")
        print(f"  costo_total={salida['costo_total']}  casos_cubiertos={salida['casos_cubiertos']}")
        print(f"  valor={salida['valor']}  tiempo={segundos:.3f} s")

===== pequena (presupuesto 6000)
base:
  estaciones=[('L1', 'incendio')]
  costo_total=5500  casos_cubiertos={'hurto': 0, 'accidente': 0, 'incendio': 3}
  valor=15  tiempo=0.000 s
propia:
  estaciones=[('L1', 'accidente'), ('L2', 'hurto'), ('L3', 'hurto')]
  costo_total=5100  casos_cubiertos={'hurto': 7, 'accidente': 4, 'incendio': 0}
  valor=19  tiempo=0.000 s
propia_ls:
  estaciones=[('L1', 'hurto'), ('L3', 'accidente'), ('L4', 'hurto')]
  costo_total=5400  casos_cubiertos={'hurto': 8, 'accidente': 4, 'incendio': 0}
  valor=20  tiempo=0.001 s
===== comuna (presupuesto 12000)
base:
  estaciones=[('L05', 'accidente'), ('L06', 'hurto'), ('L08', 'accidente'), ('L09', 'incendio'), ('L13', 'hurto'), ('L14', 'hurto'), ('L15', 'accidente')]
  costo_total=11950  casos_cubiertos={'hurto': 486, 'accidente': 296, 'incendio': 31}
  valor=1529  tiempo=0.000 s
propia:
  estaciones=[('L02', 'hurto'), ('L03', 'accidente'), ('L05', 'accidente'), ('L06', 'hurto'), ('L09', 'hurto'), ('L10', 'accidente')

#### Instancias hechas a mano donde cada heurística falla

Están en `instancias_malas/base`, `instancias_malas/propia` e `instancias_malas/propia_ls`, con los mismos cinco archivos que las instancias entregadas, a lo sumo 6 nodos y los **mismos costos base, radios y pesos oficiales** (solo cambia el presupuesto). Cada carpeta tiene un `README.md` con la traza paso a paso. La celda siguiente muestra un resumen de cada una y compara la heurística con el óptimo calculado con el Branch & Bound del proyecto (en instancias tan pequeñas los tests además lo confirman con fuerza bruta: 4^n propuestas).

In [7]:
for nombre, heuristica in HEURISTICAS.items():
    carpeta = RAIZ / "instancias_malas" / nombre
    modelo = Modelo(cargar_instancia(carpeta))
    instancia = modelo.instancia
    print(f"===== instancias_malas/{nombre}: {len(instancia.nodos)} nodos, "
          f"{len(instancia.lotes)} lotes, {len(instancia.incidentes)} incidentes, "
          f"presupuesto {modelo.presupuesto}")
    for codigo_lote, tipo in modelo.todas_las_opciones():
        cubiertos = modelo.cobertura(codigo_lote, tipo)
        if len(cubiertos) > 0:
            valor = len(cubiertos) * modelo.pesos[tipo]
            print(f"  {codigo_lote} {tipo:<10} costo={modelo.costo(codigo_lote, tipo):>5} "
                  f"cubre={cubiertos} valor={valor}")
    obtenido = heuristica(modelo)
    optimo = resolver(modelo)
    assert optimo["optimo_certificado"]
    assert obtenido["valor"] < optimo["valor"]
    print(f"  {nombre}: {obtenido['estaciones']} valor={obtenido['valor']} costo={obtenido['costo_total']}")
    print(f"  óptimo: {optimo['estaciones']} valor={optimo['valor']} costo={optimo['costo_total']}")

===== instancias_malas/base: 3 nodos, 3 lotes, 11 incidentes, presupuesto 7000
  L1 hurto      costo= 1200 cubre=[4, 5, 7, 9] valor=4
  L1 incendio   costo= 6000 cubre=[6] valor=5
  L2 hurto      costo= 1500 cubre=[1, 3, 4, 5, 7, 8, 9, 11] valor=8
  L2 accidente  costo= 3000 cubre=[2, 10] valor=6
  L2 incendio   costo= 7500 cubre=[6] valor=5
  L3 hurto      costo= 1500 cubre=[1, 3, 8] valor=3
  L3 accidente  costo= 3000 cubre=[2] valor=3
  base: [('L2', 'hurto'), ('L3', 'accidente')] valor=11 costo=4500
  óptimo: [('L1', 'hurto'), ('L2', 'accidente'), ('L3', 'hurto')] valor=13 costo=5700
===== instancias_malas/propia: 6 nodos, 4 lotes, 6 incidentes, presupuesto 5000
  L1 hurto      costo= 1200 cubre=[2] valor=1
  L1 accidente  costo= 2400 cubre=[5] valor=3
  L3 hurto      costo=  800 cubre=[2] valor=1
  L3 accidente  costo= 1600 cubre=[5] valor=3
  L3 incendio   costo= 4000 cubre=[3] valor=5
  L4 hurto      costo=  500 cubre=[1, 2, 4] valor=3
  L4 accidente  costo= 1000 cubre=[5] valor

Qué decisión lleva a cada heurística por mal camino (traza completa en cada `README.md`):

- **`base` (11 frente a 13).** En el primer paso elige `L2 hurto` porque es la que más valor agrega (8). Pero ocupa el lote L2, el único cuya estación de tránsito cubre los **dos** accidentes (peso 3 cada uno), y casi todos sus hurtos también se cubren desde L1 y L3. El óptimo es `L1 hurto` + `L2 accidente` + `L3 hurto`. La base solo mira el valor de la siguiente estación, no lo que ese lote impide construir después.
- **`propia` (6 frente a 8).** Primero elige `L4 hurto` (mejor razón, 3 / 500). En el segundo paso, entre `L3 accidente` (3 / 1600) y `L3 incendio` (5 / 4000) prefiere la estación de tránsito por su mejor razón, y con eso ocupa el último lote desde el que se llegaba al único incendio (peso 5). Elegir `L3 incendio` habría dado 8. La razón valor/costo no ve que un lote es un recurso escaso.
- **`propia_ls` (12 frente a 13).** Su mejor arranque es `L1 incendio` + `L2 hurto` (12). El óptimo, `L1 accidente` + `L2 incendio` + `L3 hurto`, exige cambiar **a la vez** las dos estaciones. Su vecindario más grande elige a mano solo una estación nueva y deja la otra al relleno goloso por razón, que siempre prefiere el CAI barato del lote libre (`L1 hurto` o `L2 hurto`) en lugar de la pareja tránsito + bomberos.

## Parte 3. La mejor propuesta posible

Obtén la propuesta de **mayor valor posible** que respete todas las reglas, para las instancias `pequena` y `comuna`. Tu programa debe garantizar que no existe una propuesta mejor, no solo encontrar una buena.

Luego compara en una tabla, para las tres instancias, el valor y el tiempo de ejecución de la heurística base, de tus heurísticas propias y de la mejor propuesta posible. En la instancia `ciudad`, explica con un cálculo por qué no reportas la mejor propuesta posible.

### Salida

Para `pequena` y `comuna`: las estaciones de la mejor propuesta (ordenadas por código de lote), su costo total, los casos cubiertos de cada tipo, su valor y el tiempo que tardó en obtenerse. Además, la tabla comparativa.

In [ ]:
casos_ej_001_parte3 = [
    {
        "entrada": {"instancia": "pequena"},
        "salida": {
            "estaciones": [("L1", "hurto"), ("L3", "accidente"), ("L4", "hurto")],
            "costo_total": 5400,
            "casos_cubiertos": {"hurto": 8, "accidente": 4, "incendio": 0},
            "valor": 20,
        },
    },
]

### Solución de la Parte 3

La mejor propuesta se obtiene con **backtracking + Branch & Bound** (`estaciones/exacto.py`). En el nivel *i* del árbol se decide el lote *i*: no construir nada o construir una de sus estaciones. En vez de recorrer los 4^n caminos, se poda una rama cuando se puede demostrar que no supera a la mejor propuesta ya encontrada:

- **Cota barata**: aunque se construyera todo lo que queda, no se cubre más que la unión de coberturas de los lotes restantes.
- **Cota de mochila fraccional**: a cada estación restante se le da su ganancia marginal sola, se ignora la regla de un lote, una estación y se permite comprar "un pedazo" de la última. Relajar reglas solo puede subir el máximo, así que la cota nunca subestima; por eso podar con ella no pierde el óptimo.
- **Arranque**: la búsqueda empieza con la propuesta de `propia_ls` como mejor conocida, para podar desde el primer nodo.

**Garantía**: si la búsqueda termina sin cortarse por tiempo, se recorrió (explícitamente o por poda demostrada) todo el árbol, y la respuesta es el óptimo (`optimo_certificado = True`). Los tests lo confirman con fuerza bruta en `pequena` y en 30 instancias aleatorias pequeñas. Entre propuestas de igual valor se queda con la de menor costo.

In [8]:
def parte3(entrada):
    """Mejor propuesta posible, solo si el Branch & Bound la certifica como óptima."""
    resultado = resolver(modelo_de(entrada["instancia"]))
    assert resultado["optimo_certificado"], "la búsqueda no terminó: no es un óptimo garantizado"
    return salida_del_enunciado(resultado)


for caso in casos_ej_001_parte3:
    obtenido = parte3(caso["entrada"])
    assert obtenido == caso["salida"], (caso["entrada"], obtenido)
print("OK: el caso de ejemplo de la Parte 3 coincide")

OK: el caso de ejemplo de la Parte 3 coincide


In [9]:
OPTIMOS = {}
for instancia in ("pequena", "comuna"):
    resultado = resolver(modelo_de(instancia))
    OPTIMOS[instancia] = resultado
    salida = salida_del_enunciado(resultado)
    print(f"===== {instancia}")
    print(f"  estaciones={salida['estaciones']}")
    print(f"  costo_total={salida['costo_total']}  casos_cubiertos={salida['casos_cubiertos']}")
    print(f"  valor={salida['valor']}  tiempo={resultado['tiempo_s']:.3f} s")
    print(f"  optimo_certificado={resultado['optimo_certificado']}  estadisticas={resultado['estadisticas_bb']}")

===== pequena
  estaciones=[('L1', 'hurto'), ('L3', 'accidente'), ('L4', 'hurto')]
  costo_total=5400  casos_cubiertos={'hurto': 8, 'accidente': 4, 'incendio': 0}
  valor=20  tiempo=0.002 s
  optimo_certificado=True  estadisticas={'nodos': 51, 'podas_presupuesto': 34, 'podas_cota': 20, 'valor_arranque': 20}
===== comuna
  estaciones=[('L03', 'accidente'), ('L06', 'hurto'), ('L09', 'hurto'), ('L11', 'hurto'), ('L13', 'accidente'), ('L14', 'hurto'), ('L15', 'incendio')]
  costo_total=11700  casos_cubiertos={'hurto': 519, 'accidente': 300, 'incendio': 31}
  valor=1574  tiempo=1.263 s
  optimo_certificado=True  estadisticas={'nodos': 41493, 'podas_presupuesto': 2283, 'podas_cota': 29799, 'valor_arranque': 1574}


In [10]:
def texto_de_celda(valor, segundos):
    """'valor (tiempo s)' para la tabla comparativa."""
    return f"{valor} ({segundos:.3f} s)"


cabecera = f"{'instancia':<10}"
for nombre in ("base", "propia", "propia_ls", "óptimo"):
    cabecera += f"{nombre:>22}"
print(cabecera + f"{'precálculo':>13}")
for instancia in ("pequena", "comuna", "ciudad"):
    celdas = []
    for nombre in ("base", "propia", "propia_ls"):
        resultado, segundos = RESULTADOS[(instancia, nombre)]
        celdas.append(texto_de_celda(resultado["valor"], segundos))
    if instancia in OPTIMOS:
        optimo = OPTIMOS[instancia]
        celdas.append(texto_de_celda(optimo["valor"], optimo["tiempo_s"]))
    else:
        celdas.append("no calculado")
    fila = f"{instancia:<10}"
    for celda in celdas:
        fila += f"{celda:>22}"
    print(fila + f"{SEGUNDOS_DE_PRECALCULO[instancia]:>11.2f} s")
print()
print("Los tiempos de cada método NO incluyen el precálculo (lectura + Dijkstra + costos de la")
print("Parte 1), que se mide aparte en la última columna y se hace una sola vez por instancia.")
print("El tiempo del óptimo sí incluye su arranque con propia_ls.")

instancia                   base                propia             propia_ls                óptimo   precálculo
pequena             15 (0.000 s)          19 (0.000 s)          20 (0.001 s)          20 (0.002 s)       0.00 s
comuna            1529 (0.000 s)        1492 (0.001 s)        1574 (0.072 s)        1574 (1.263 s)       0.07 s
ciudad            7591 (0.009 s)        7706 (0.013 s)       7827 (33.737 s)          no calculado       0.66 s

Los tiempos de cada método NO incluyen el precálculo (lectura + Dijkstra + costos de la
Parte 1), que se mide aparte en la última columna y se hace una sola vez por instancia.
El tiempo del óptimo sí incluye su arranque con propia_ls.


In [11]:
modelo = modelo_de("ciudad")
cantidad_lotes = len(modelo.instancia.lotes)
opciones_por_lote = len(modelo.tipos) + 1
propuestas = opciones_por_lote ** cantidad_lotes
segundos_por_anio = 365.25 * 24 * 3600
anios = propuestas / 10**9 / segundos_por_anio
print(f"Cada lote: no construir o construir una de {len(modelo.tipos)} estaciones = {opciones_por_lote} opciones.")
print(f"Propuestas posibles en ciudad: {opciones_por_lote}^{cantidad_lotes} = {propuestas}")
print(f"(unas {propuestas:.3e}, un número de {len(str(propuestas))} cifras)")
print(f"Revisando 10^9 propuestas por segundo: {anios:.3e} años.")
print(f"La edad del universo es unos 1.38e10 años: harían falta {anios / 1.38e10:.1e} universos.")

Cada lote: no construir o construir una de 3 estaciones = 4 opciones.
Propuestas posibles en ciudad: 4^80 = 1461501637330902918203684832716283019655932542976
(unas 1.462e+48, un número de 49 cifras)
Revisando 10^9 propuestas por segundo: 4.631e+31 años.
La edad del universo es unos 1.38e10 años: harían falta 3.4e+21 universos.


Por eso en `ciudad` no se reporta la mejor propuesta posible: la fuerza bruta es imposible, y la poda del Branch & Bound **no garantiza terminar** en un tiempo razonable (en el peor caso sigue siendo exponencial). Con `py main.py parte3 ciudad --limite 20` la búsqueda se corta sin certificar: devuelve la mejor propuesta encontrada, que no es un óptimo garantizado. Para `ciudad` se usan las heurísticas.

## Parte 4. Propuesta para el concejo

Elige la propuesta que recomendarías para la instancia `ciudad`, obtenida con cualquiera de tus heurísticas, y preséntala con:

1. **Un mapa** de la ciudad, hecho con `folium` o con cualquier otra librería, donde se distingan los lotes no usados, las estaciones construidas según su tipo, y los incidentes cubiertos y no cubiertos. Mostrar el resultado de forma gráfica sobre un mapa es obligatorio para todos los equipos, participen o no en la feria.
2. **Una tabla** con cada estación construida: lote, tipo, costo y casos que cubre. Al final, el costo total frente al presupuesto, los casos cubiertos de cada tipo y el valor de la propuesta.
3. **Un párrafo** que justifique por qué recomiendas esa propuesta y no otra de las que obtuviste.

Repite el mapa y la tabla para la mejor propuesta posible de la instancia `comuna`.

### Solución de la Parte 4

El mapa lo genera el mismo exportador que usa `py main.py parte4` (`estaciones/mapa.py`, función `exportar_html`): un archivo HTML autocontenido que se abre con doble clic. El mapa solo **dibuja** resultados ya calculados por el paquete (Leaflet no calcula rutas, distancias ni combinaciones). Se distinguen los lotes no usados, las estaciones construidas por tipo (forma y color) y los incidentes cubiertos y no cubiertos. El fondo de calles de OpenStreetMap necesita internet.

En la tabla, "casos" es la cantidad de incidentes que cubre esa estación por sí sola; los totales cuentan cada incidente una sola vez.

In [12]:
def guardar_y_mostrar_mapa(instancia, metodo, resultado, detalle):
    """Escribe el mapa en salidas/ con el exportador del proyecto y lo muestra si se puede."""
    modelo = modelo_de(instancia)
    fechas = leer_fechas(carpeta_de_instancia(instancia))
    documento = exportar_html(modelo, resultado, fechas,
                              titulo=f"Estaciones · {instancia}", detalle=detalle)
    ruta_relativa = Path("salidas") / f"mapa_{instancia}_{metodo}.html"
    ruta = RAIZ / ruta_relativa
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(documento, encoding="utf-8")
    print(f"Mapa guardado en {ruta_relativa.as_posix()} (ábrelo con doble clic)")
    try:
        from IPython.display import IFrame, display
    except ImportError:
        return
    display(IFrame(src=ruta_relativa.as_posix(), width="100%", height=560))

In [13]:
resultado_ciudad, segundos_ciudad = RESULTADOS[("ciudad", "propia_ls")]
guardar_y_mostrar_mapa("ciudad", "propia_ls", resultado_ciudad, "propia_ls")
print()
print(tabla_texto(modelo_de("ciudad"), resultado_ciudad))

Mapa guardado en salidas/mapa_ciudad_propia_ls.html (ábrelo con doble clic)

lote    tipo       estación                           costo   casos
L01     accidente  Estación de tránsito y grúas        2600     234
L03     accidente  Estación de tránsito y grúas        1600     121
L04     hurto      CAI                                  850     355
L08     hurto      CAI                                 1000     240
L11     hurto      CAI                                 1000     261
L12     accidente  Estación de tránsito y grúas        1700     153
L14     hurto      CAI                                 1100      84
L16     accidente  Estación de tránsito y grúas        2100     263
L19     accidente  Estación de tránsito y grúas        1600     233
L25     accidente  Estación de tránsito y grúas        1700      89
L28     incendio   Estación de bomberos                4000      74
L29     incendio   Estación de bomberos                4250      64
L34     accidente  Estación de tránsito

In [14]:
modelo = modelo_de("ciudad")
for nombre in ("base", "propia", "propia_ls"):
    resultado, segundos = RESULTADOS[("ciudad", nombre)]
    print(f"{nombre:<10} valor={resultado['valor']:>5}  costo={resultado['costo_total']:>6} de {modelo.presupuesto}  "
          f"estaciones={len(resultado['estaciones']):>2}  casos={resultado['casos_cubiertos']}  {segundos:.2f} s")

base       valor= 7591  costo= 39850 de 40000  estaciones=20  casos={'hurto': 1350, 'accidente': 1757, 'incendio': 194}  0.01 s
propia     valor= 7706  costo= 39950 de 40000  estaciones=23  casos={'hurto': 1887, 'accidente': 1718, 'incendio': 133}  0.01 s
propia_ls  valor= 7827  costo= 40000 de 40000  estaciones=23  casos={'hurto': 1768, 'accidente': 1798, 'incendio': 133}  33.74 s


**Recomendación al concejo (instancia `ciudad`).** Recomendamos la propuesta de **`propia_ls`**: 23 estaciones (11 de tránsito y grúas, 10 CAI y 2 de bomberos) que cuestan **40.000 millones, exactamente el presupuesto**, y cubren 1.768 hurtos, 1.798 accidentes y 133 incendios, con un **valor de 7.827**. Es la de mayor valor de las tres que obtuvimos: la heurística base de Planeación logra 7.591 (20 estaciones, 39.850 millones) y nuestra heurística `propia` 7.706 (23 estaciones, 39.950 millones); es decir, 236 puntos más que la base (+3,1 %) y 121 más que `propia` (+1,6 %) con el mismo dinero. Frente a la base, cubre 418 hurtos y 41 accidentes más a cambio de 61 incendios menos (la base construye 3 estaciones de bomberos, nosotros 2); con los pesos oficiales (1, 3 y 5) ese cambio suma 418 + 123 − 305 = 236. Además es **determinista**: no usa azar, así que cualquiera puede repetir el cálculo y obtener exactamente la misma propuesta, y su costo de cálculo (entre 20 y 35 s según el equipo) es despreciable frente a una inversión de 40.000 millones. Debemos ser honestos sobre su límite: **no podemos certificar que sea la mejor propuesta posible** para `ciudad`, porque hay 4^80 ≈ 1,46 × 10^48 propuestas y nuestro Branch & Bound no termina en un tiempo razonable; no se calculó ni se probó el óptimo de `ciudad`. Lo único que tenemos es un indicio, no una prueba: en `pequena` y `comuna`, donde sí certificamos el óptimo, `propia_ls` lo alcanzó exactamente. Si el concejo quisiera dar más prioridad a los incendios, basta con cambiar los pesos (por ejemplo en la interfaz) y recalcular.

In [15]:
resultado_comuna = OPTIMOS["comuna"]
guardar_y_mostrar_mapa("comuna", "exacto", resultado_comuna, "exacto · óptimo certificado")
print()
print(tabla_texto(modelo_de("comuna"), resultado_comuna))

Mapa guardado en salidas/mapa_comuna_exacto.html (ábrelo con doble clic)

lote    tipo       estación                           costo   casos
L03     accidente  Estación de tránsito y grúas        1800     222
L06     hurto      CAI                                  850     180
L09     hurto      CAI                                  850     166
L11     hurto      CAI                                 1050     200
L13     accidente  Estación de tránsito y grúas        2100     213
L14     hurto      CAI                                  800     159
L15     incendio   Estación de bomberos                4250      31
-------------------------------------------------------------------
costo_total=11700 de presupuesto=12000
casos_cubiertos={'hurto': 519, 'accidente': 300, 'incendio': 31}
valor=1574


## Notas de interpretación y límites

- **Redondeo del costo.** "Al millón más cercano" se implementa con `.5` hacia arriba (`ROUND_HALF_UP` de `decimal`), la convención escolar. El `round()` de Python redondea `.5` al par (2.5 → 2). En las tres instancias entregadas no hay ningún costo con `.5` exacto, así que no cambia ningún resultado; en una instancia nueva sí podría.
- **Orden de los códigos de lote.** "Menor código de lote" se toma en **orden natural**: `L2` va antes que `L10`. Con códigos con ceros (`L01`…`L15`) es igual al orden de texto. Las estaciones se listan siempre en ese orden.
- **Bordes.** Un incidente a distancia exactamente igual al radio queda cubierto (`<=`) y una propuesta que gasta exactamente el presupuesto es válida (`<=`).
- **Uso de `decimal`.** Solo se usa para leer números escritos como texto sin errores de redondeo (las longitudes se vuelven enteros exactos, por ejemplo décimas de metro) y para redondear el costo. No calcula rutas, distancias en grafos ni combinaciones: las distancias las calcula el Dijkstra propio con `heapq`.
- **Datos escritos a mano.** La lectura acepta coma decimal (`367,9`) y tipos de incidente con mayúsculas o espacios (`"Hurto "`). Los datos mal formados se rechazan con un mensaje que dice qué archivo y qué dato fallan.
- **`propia_ls` es híbrida**: arranques golosos de una estación a la vez + mejora local (quitar y cambiar estaciones). El requisito 2.2 se cumple con `propia`, que es constructiva pura.
- **Tiempos.** Los de las heurísticas y el óptimo no incluyen el precálculo de la Parte 1 (medido aparte: menos de 1 s en `ciudad`). `propia_ls` en `ciudad` tarda entre 20 y 35 s según el equipo. Varían un poco entre ejecuciones y equipos.
- **Instancias de la sustentación.** Todo lee cualquier carpeta con los cinco archivos: en este notebook basta con pasar su ruta, por ejemplo `parte2({"instancia": r"C:\ruta\nueva", "heuristica": "propia_ls"})`; en consola, `py main.py parte2 C:\ruta\nueva propia_ls`. `py main.py parte4` elige solo el método: con más de 30 lotes usa `propia_ls` y si no, el óptimo exacto (con límite de tiempo).
- **Interfaz de la feria.** `py main.py interfaz` abre una página local (`http://127.0.0.1:8765`) para elegir la instancia, cambiar el presupuesto, los pesos y el método, y ver la propuesta sobre el mapa. Usa solo la biblioteca estándar.